# ICS 604: APPLIED DATA SCIENCE

## Data Wrangling: Data Preparation and Cleaning

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## About the Data

The dataset `spending_clean_ex.csv` was specifically constructed from the Medicare dataset. Rather than using the raw Medicare data directly, this dataset was intentionally modified to include realistic data-cleaning use cases. These examples are designed to help illustrate typical issues—such as missing values, inconsistencies, or formatting problems—that analysts often need to address when working with real-world data.

In [ ]:
spending_df = pd.read_csv('data/spending_clean_ex.csv', index_col='unique_id')
spending_df.head(7)

In [ ]:
spending_df.shape

## Inspecting and Modifying Data Types

Recall that the `dtypes` attribute returns the data type of each column in a DataFrame. Inspecting the output for `spending_df` reveals potential data type issues. In particular, the `doctor_id` column is stored as an `int64`, even though it represents an identifier rather than a numeric quantity, and the `spending` column is stored as an `object`, even though it should represent numeric values.

In [ ]:
spending_df.dtypes

These misidentified data types can lead to incorrect analysis or inefficient computation, so they need to be corrected. To fix this, we must explicitly cast each affected column to an appropriate Pandas data type using the `astype()` method. For example, we can convert the `doctor_id` column to an `object` type:

```python
>>> spending_df['doctor_id'].astype('object')
```

As with most Pandas operations, `astype()` is not performed in place. This means that simply calling the method does not permanently modify the DataFrame. To make the change persistent, we must explicitly overwrite the original column with the converted version.

In [ ]:
spending_df['doctor_id'].astype('object')
spending_df.dtypes

### Changing Column Data Type

After performing this conversion and updating the DataFrame, inspecting `spending_df.dtypes` confirms that the change has taken effect:

In [ ]:
spending_df['doctor_id'] = spending_df['doctor_id'].astype('object')
spending_df.dtypes

After this update, the `doctor_id` column is correctly stored as an object, while the remaining columns retain their respective data types. This reinforces an important Pandas concept: **data type conversions are not performed in place**, so the converted values must be explicitly reassigned to the DataFrame to make the change permanent.

## String Methods on Series of Type `object`

A Series with data type `object` (commonly used to store text data) provides a special string accessor called `.str`. This accessor allows you to work with the individual string values inside the Series in a convenient and consistent way. For example, using 

```python
spending_df["spending"].str 
```

creates a string-aware view of the Series, enabling string-specific operations.

The `.str` accessor exposes many of the same methods that are available on standard Python string objects, such as `lower()`, `upper()`, `contains()`, and `replace()`. The key difference is that these methods are applied element-wise across the entire Series. Instead of operating on a single string and returning a single value, each method processes every entry in the Series and returns a new Series with the transformed results.

It is important to note that string methods are not called directly on the Series itself. Instead, they must be accessed through the `.str` attribute. This design makes it explicit that the operation is a vectorized string operation, ensuring clarity and efficiency when performing text manipulation on Series data.

 <img src="https://www.dropbox.com/scl/fi/vj8v9ha7ac6ywzbvt9kmm/str_upper.png?rlkey=ac4zlgcrzwpbda0zpsbx3yrim&st=h3wpgbe1&dl=1">

In [ ]:
some_series = pd.Series(["Hi", "there", "How", "are", "you"])
some_series.str.upper()

### Using `replace()` to Fix Column's Data

Often, numeric values stored as strings contain extra characters that prevent them from being used directly for calculations. In this case, the `spending` column includes formatting symbols such as the dollar sign (`$`) and commas (`,`), which must be removed before the data can be treated as numeric.

The `replace()` string method can be used to clean these values. By replacing the `"$"` character with an empty string (`""`), we effectively remove it from every entry in the Series. The same approach applies to commas: replacing `","` with `""` converts values like `3,454,420.29` into `3454420.29`. When used through the `.str` accessor, `replace()` is applied element-wise to all values in the column.

For example, 

```python
spending_df["spending"].str.replace("$", "") 
```
returns a new Series with the dollar signs removed. However, this operation does not modify the original column in place. To make the changes permanent, the cleaned Series must be assigned back to the `spending` column, overwriting the existing data.

In [ ]:
spending_df.sort_values(by="spending").head(10)

In [ ]:
spending_df["spending"] = spending_df["spending"].str.replace("$", "")
spending_df["spending"] = spending_df["spending"].str.replace(",", "")
spending_df["spending"] = spending_df["spending"].astype('float64')
spending_df.head()

In [ ]:
spending_df.dtypes

### Combining Multiple String Operations

When multiple string transformations are required, **method chaining** allows us to combine them into a single, readable operation. Instead of applying each transformation in separate steps, chaining groups related functionality together, making the code easier to understand, maintain, and modify.

Each call in the chain operates on the result of the previous step. Because string operations in pandas are accessed through the `.str` accessor, we must call `.str` on each resulting Series before applying another string method.

In the example shown below, the original `spending` column is overwritten with the cleaned and converted data. The single statement removes formatting characters and converts the values to floating-point numbers, producing a column that is ready for numerical analysis.

In [ ]:
spending_df = pd.read_csv('data/spending_clean_ex.csv', index_col='unique_id')

spending_df["spending"] = (spending_df["spending"]
                           .str.replace("$", "")
                           .str.replace(",", "")
                           .astype("float64"))
spending_df.dtypes

In [ ]:
spending_df = pd.read_csv('data/spending_clean_ex.csv', index_col='unique_id')

spending_df["spending"] = (spending_df["spending"]
                           .str.replace(["$", ","], "")
                           .astype("float64"))
spending_df.dtypes

## Missing Data

Missing data is a common issue in real-world datasets and can significantly complicate analysis or lead to misleading conclusions if not handled properly. Calculations, summaries, and models may behave unexpectedly when values are absent, making it essential to identify and address missing data early in the data-cleaning process.

There are many reasons why data may be missing. Human error can result in values being deleted or entered incorrectly. In other cases, data collection may be incomplete—for example, an early round of data collection might omit a field such as salary, while a later round includes it. Technical issues, such as defective instruments or intermittent communication failures, can also prevent values from being recorded.

Missing data is not always represented as an actual blank or null value. Often, datasets use sentinel values to indicate missing information. Common examples include numeric placeholders like `999999` or text-based indicators such as `"UNKNOWN"` or `"N/A"`. Recognizing these sentinels is crucial, as they must typically be converted to proper missing values before meaningful analysis can be performed.

### Handling Missing Data

Handling missing data begins with correctly identifying which values are truly missing, including both explicit nulls and any sentinel values used to represent absence. Once missing values have been identified, an appropriate strategy must be chosen to deal with them.

In practice, there are two main approaches to handling missing data. The first is **filtering**, where rows or columns containing missing values are removed from the dataset. The second is **imputation**, where missing values are filled in with reasonable substitute values. Each approach has trade-offs and can affect the results of an analysis in different ways.

Choosing between filtering and imputation is highly application- and data-dependent. Key considerations include whether you can afford to discard data without losing important information and whether there is a sensible way to estimate the missing values. Simple imputation methods might use default values, such as the mean of a numeric feature, while more advanced approaches may rely on statistical or machine learning models to infer likely values.


### Identifying Missing Values 

Missing values can appear in datasets in several different forms, and recognizing them correctly is essential for effective data analysis. One common form is an empty field in the data source. When data is loaded into pandas, these empty fields are automatically interpreted as `NaN` (Not a Number), which is pandas’ standard representation for missing data.

The `NaN` value originates from NumPy and is represented as a special floating-point value that is not equal to any other number — including itself. (E.g., `np.nan != np.nan` evaluates to `True`.) Because of this unique behavior, special functions are required to detect and handle `NaN` values correctly.
  
Another common way missing data appears is through sentinel values. Sentinels are placeholder values inserted for convenience to indicate missing information, such as using `999999` for a ZIP code or `0` or `-1` for a salary. While these values make missing data explicit, they are not automatically treated as missing by pandas and must be handled manually. 

Identifying and converting sentinel values is critical for accurate modeling and analysis. Leaving sentinel values in the dataset can significantly skew results — for example, including a sentinel salary value when computing the mean will distort the outcome. Similarly, failing to handle `NaN` values can cause calculations to propagate missing results. A common practice is to replace all sentinel values (such as `999999`) with `np.nan`, allowing pandas’ built-in missing-data handling tools to work correctly.

In [ ]:
np.nan == np.nan

In [ ]:
x = pd.Series([10, np.nan, 4])
x

In [ ]:
_sum = sum(x)
_mean = np.mean(x) # did not work in previous versions :(
nanmean = np.nanmean(x)  # computes the arithmetic mean ignoring NaNs

# different ways of printing  

print("The sum is {}".format(_sum))
print(f"The mean is {_mean}")
print("The nanmean is %s" % (nanmean))

In [ ]:
int(np.nan)

In Pandas, missing values can be easily detected using the `isnull()` or `isna()` methods, which are available on both Series and DataFrame objects. These methods scan the data and return a Boolean result for each entry: `True` if the value is `NaN` (i.e., missing) and `False` otherwise.

This functionality provides a simple way to locate missing data, enabling further steps such as filtering out incomplete rows, counting missing values, or preparing the dataset for imputation. Since `isnull()` and `isna()` are equivalent, you can use whichever name you find more intuitive in your workflow.

In [ ]:
spending_df.head()

In [ ]:
spending_df.isnull().head()

In [ ]:
spending_df.isna().head()

### Counting Missing Values

Counting missing values in a dataset is an important step to understand the extent of incomplete data. In pandas, this can be done efficiently using the Boolean output of `isnull()` or `isna()`. Since Python treats `True` as `1` and `False` as `0`, applying the `sum()` function to the Boolean Series gives the total number of missing values. For example,

```python
>>> spending_df["spending"].isnull().sum()
```

returns the count of `NaN` entries in the `spending` column.

An alternative approach is to use the `size` and `count()` attributes of a Series. The `size` attribute gives the total number of elements, including missing values, while `count()` returns the number of non-missing entries. Subtracting `count()` from size yields the number of missing values. This method provides the same result as summing the Boolean mask but can be more intuitive when working with both Series and DataFrames.

```python
>>> spending_df["spending"].size - spending_df["spending"].count()
```

In [ ]:
spending_df['spending'].isnull().sum()

In [ ]:
print(spending_df["spending"].size - spending_df["spending"].count())

In [ ]:
spending_df.isnull().sum()

In [ ]:
spending_df.count()

In [ ]:
spending_df.shape[0] - spending_df.count()

### Changing Column Data Type

Changing the data type of a column in pandas can be done easily **only when the existing data is compatible** with the target type. For example, a column containing last names cannot be converted to a numeric type like `float64`, because the text entries cannot be interpreted as numbers.

Missing values require special attention. Columns that contain `NaN` cannot be directly converted to integer types (`int64`), since `NaN` is inherently a floating-point value. Attempting such a conversion will raise an error. On the other hand, converting a column with `NaN` values to a string type (`str`) is allowed, but the `NaN` entries are transformed into the literal string `"nan"`.

In [ ]:
x = pd.DataFrame([[1, "10"], [2, np.nan], [3, "4"]])
x[1].astype('int')

In [ ]:
x = pd.DataFrame([[1, "10"], [2, np.nan], [3, "4"]])
display(x)

print(x.dtypes)
print("-" * 10)
x[1].astype('float64')

In [ ]:
y = x[1].astype('str')

display(y)
print(y[1], type(y[1]))

The `nb_beneficiaries` column cannot be directly converted to `int64` because it contains missing values. Attempting this conversion in pandas triggers an error:

```python
>>> spending_df['nb_beneficiaries'] = spending_df['nb_beneficiaries'].astype('int64')
...
IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer
```

In [ ]:
spending_df['nb_beneficiaries'] = spending_df['nb_beneficiaries'].astype('int64')

To perform the conversion safely, the missing values must be handled first. Only after addressing the missing data can the column be converted without errors.

### Filtering Missing Values

One straightforward way to handle missing values is to filter them out by dropping entries that contain them. This approach is simple, but whether it is appropriate depends on the dataset and the analysis. For example, removing a handful of missing values from a dataset of one million entries is unlikely to cause issues. However, if all missing values occur for a specific group—such as all males lacking salary information—dropping these rows could introduce serious bias into a market segmentation model.

Filtering can be done either by **subsetting with a Boolean Series** or using the `dropna()` method. For instance, you can create a Boolean mask identifying missing entries with `isnull()`, invert it with `~` to select non-missing rows, and then subset the DataFrame:

In [ ]:
data = pd.DataFrame({"A":[1, 2, 3, 4, 5, 6], "B":[10, 2, 5, 7, np.nan, 19]})
data

In [ ]:
is_null_values = data["B"].isnull()
is_null_values

In [ ]:
data[~is_null_values]

In [ ]:
null_entries = spending_df["spending"].isnull()
not_null_entries = ~null_entries
spending_no_na_df = spending_df[not_null_entries]

print(spending_df.shape, spending_no_na_df.shape)

The above can also be written more concisely in a single line:

In [ ]:
spending_no_na_df = spending_df[~spending_df["spending"].isnull()]

print(spending_df.shape, spending_no_na_df.shape)

When subsetting with a Boolean Series, it is important that the mask has the same length as the axis being indexed. Otherwise, pandas will raise an error because it cannot align the mask with the data.

#### Filtering `NaN` Using the `dropna()` Method 

Pandas provides a convenient method, `dropna()`, for filtering out missing values (`NaN`) from a DataFrame. By default, `dropna()` examines all columns and removes any row that contains at least one `NaN` value. This makes it a quick way to clean a dataset of incomplete entries, especially when missing data is sparse.

The method also allows more flexibility. You can specify a subset of columns to check, so that only rows with missing values in certain critical columns are dropped, leaving other rows intact. Additionally, `dropna()` can operate along different axes: using `axis=0` drops rows containing `NaN` values (the default behavior), while `axis=1` drops columns that contain any missing values.

  
<img src="https://www.dropbox.com/scl/fi/fzuvtea9va7hudj9q799z/axis_drop.png?rlkey=5jwc6xhs1ccelkw3bcb8h6j5u&st=kwkd8bai&dl=1" alt="drawing" style="width:500px">

In [ ]:
# This is when small test cases are useful

test_df = pd.DataFrame({"X": [1, 2, 3, 4], "Y": ['A', 'E', 'C', np.nan]})
test_df

In [ ]:
test_df.dropna()

In [ ]:
test_df.dropna(axis=1)

In [ ]:
# dropna() is not in-plance operation

test_df

In [ ]:
test_df.dropna(axis=0, subset=["X"])

In [ ]:
test_df.dropna(axis=0, subset=["Y"])

In [ ]:
test_df.dropna(subset=["Y"])

In [ ]:
# dropna() is not in-plance operation

test_df

### Filling Missing Values

Filling missing values, often called *missing value imputation* or simply *imputation*, is a crucial step in data cleaning, especially when working with limited datasets. Proper imputation allows analyses to proceed without losing valuable information due to incomplete entries.

There are two main strategies for imputing missing values. The first is to fill in a **representative constant**, such as a default value, the mean, or the median of a column. The second approach is **dynamic imputation**, where missing values are estimated based on surrounding data or patterns. For example, the number of cars passing through the University Ave's H1 exit this Wednesday might be approximated using the count from last Wednesday, while a missing temperature reading at 2 PM could be estimated as the average of the values at 1:55 PM and 2:05 PM.

Both constant and dynamic imputation strategies can be implemented in pandas using the `fillna()` method, which provides a flexible way to replace `NaN` values with meaningful substitutes.

#### `fillna()` with Static Values 

The `fillna()` method can be used to replace missing values with **static (constant) values**. One simple approach is to provide a single scalar value, which will be used to fill all `NaN` entries across the entire DataFrame. For example,

```python
>>> filled_spending_df = spending_df.fillna(0)
```
replaces every missing value with `0`, regardless of the column.

For more control, `fillna()` also accepts a **dictionary** that maps column names to replacement values. This allows different columns to be filled with values that make sense for their data types and semantics. In the example below, missing values in the `specialty` column are replaced with `"UNKNOWN"`, while missing values in `nb_beneficiaries` and `spending` are filled with `0`. This column-specific approach is often preferable, as it preserves meaning and avoids introducing inappropriate default values.

```python
filled_spending_df = spending_df.fillna(
    {"specialty": "UNKNOWN", 
     "nb_beneficiaries": 0, 
     "spending": 0}
)
```

In [ ]:
spending_df.head(10)

In [ ]:
filled_spending_df = spending_df.fillna(0)
filled_spending_df.head(10)

In [ ]:
filled_spending_df = spending_df.fillna(
    {"specialty": "UNKNOWN", 
     "nb_beneficiaries": 0, 
     "spending": 0}
)
filled_spending_df.head(10)

#### `fillna()` with Dynamically Computed Values

Dynamic value imputation refers to filling missing entries with values that are **derived from the existing data**, rather than using fixed constants. This approach can better reflect the underlying patterns in the dataset and is often preferred when sufficient data is available.

There are many strategies for dynamic imputation. One common example is replacing missing values with the mean of the corresponding column. In the code shown, the average spending and the average number of beneficiaries are first computed using the `mean()` method. These averages are then supplied to `fillna()` as a dictionary, allowing each column to be filled with an appropriate, data-driven value.

```python
>>> average_spending = spending_df["spending"].mean()
>>> average_nb_beneficiaries = spending_df["nb_beneficiaries"].mean()
>>> filled_spending_df = spending_df.fillna( {"nb_beneficiaries": average_nb_beneficiaries, 
                      "spending": average_spending} )
```

In [ ]:
average_spending = spending_df["spending"].mean()
average_nb_beneficiaries = spending_df["nb_beneficiaries"].mean()
filled_spending_df = spending_df.fillna( {"nb_beneficiaries": average_nb_beneficiaries, 
                                          "spending": average_spending} )
filled_spending_df.head(10)

#### Forward and Backward Filling Missing Values

In some datasets — especially time-ordered or sequential data — missing values can be filled using information from neighboring observations. Pandas provides two convenient methods for this purpose: **backward fill** and **forward fill**.

The `bfill()` (backward fill) method replaces a missing value with the next non-missing value that appears **after** it in the data. In contrast, the `ffill()` (forward fill) method fills a missing value using the most recent non-missing value that comes **before** it. These approaches assume that nearby values are reasonable approximations, which is often valid for slowly changing measurements such as sensor readings or time-series data.

Another common strategy is to fill missing values by **averaging the previous and next valid values**. This approach can provide a smoother estimate when the data is expected to change gradually, combining information from both directions rather than relying on a single neighboring value.

In [ ]:
complete_data = pd.Series([10, 20, 20, 5, 30, 100, 30, 20, 10, 10, 
                           20, 10, 5, 20, 50, 40, 50, 50, 50, 50, 60])
test_df = pd.Series([10, 20, 20, 5, np.nan, 100, 30, 20, 10, 10, 
                     20, 10, 5, np.nan, 50, 40, 50, 50, 50, 50, 60])
data = pd.DataFrame({"Complete": complete_data, "test_df":test_df})

data

In [ ]:
ffilled_data = data.ffill()
ffilled_data

In [ ]:
bfilled_data = data.bfill()
bfilled_data

In [ ]:
ffilled_data.plot(xticks=ffilled_data.index, grid=True)

In [ ]:
bfilled_data.plot(xticks=bfilled_data.index, grid=True)

#### `fillna()` with Dynamic Values -- Cont'd

There are many possible strategies for dynamically imputing missing values, ranging from simple to highly sophisticated. In addition to basic techniques like using the mean, more structured approaches — such as regression models — can be used to predict missing values based on other features in the dataset. Even more advanced imputation methods rely on machine learning models that learn complex relationships within the data.

When individual entries are similar to their neighbors, simpler strategies can be very effective. For example, copying the value from the previous or next instance works well for ordered or time-based data. Another common approach is to average values from similar observations using techniques like `k-nearest neighbors (KNN)`, which leverage similarity across multiple features.

Ultimately, imputation is flexible and context-dependent. You are not limited to predefined methods — custom logic or domain-specific models can often provide the most meaningful replacements, as long as the assumptions behind the chosen approach are well understood.